In [1]:
!pip3 install pandas numpy bokeh ipywidgets

   ---------------------------------------- 0.0/6.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/6.4 MB ? eta -:--:--
   --- ------------------------------------ 0.5/6.4 MB 1.1 MB/s eta 0:00:06
   ---- ----------------------------------- 0.8/6.4 MB 1.1 MB/s eta 0:00:06
   ------ --------------------------------- 1.0/6.4 MB 1.2 MB/s eta 0:00:05
   --------- ------------------------------ 1.6/6.4 MB 1.4 MB/s eta 0:00:04
   --------- ------------------------------ 1.6/6.4 MB 1.4 MB/s eta 0:00:04
   ------------- -------------------------- 2.1/6.4 MB 1.4 MB/s eta 0:00:04
   ---------------- ----------------------- 2.6/6.4 MB 1.5 MB/s eta 0:00:03
   ----------------- ---------------------- 2.9/6.4 MB 1.5 MB/s eta 0:00:03
   ------------------- -------------------- 3.1/6.4 MB 1.5 MB/s eta 0:00:03
   ---------------------- ----------------- 3.7/6.4 MB 1.5 MB/s eta 0:00:02
   -------------------------- ---

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\Tharushi Ridmika\\AppData\\Local\\Programs\\Python\\Python312\\share\\jupyter\\labextensions\\@jupyter-widgets\\jupyterlab-manager\\static\\vendors-node_modules_d3-color_src_color_js-node_modules_d3-format_src_defaultLocale_js-node_m-09b215.2643c43f22ad111f4f82.js'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
!pip3 install bokeh

  Using cached bokeh-3.9.2-py3-none-any.whl.metadata (10 kB)
  Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
Using cached bokeh-3.9.2-py3-none-any.whl (6.4 MB)
Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl (226 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:

import numpy as np
import pandas as pd
import bokeh

from bokeh.io import output_notebook, show
from bokeh.models import (
    ColorBar,
    ColumnDataSource,
    CustomJS,
    HoverTool,
    LinearColorMapper,
)
from bokeh.models.widgets import Select, Slider
from bokeh.palettes import Plasma256
from bokeh.plotting import figure
from bokeh.layouts import column, row


output_notebook()


def lat_lon_to_mercator(lat, lon):
    r_major = 6378137.0
    x = r_major * np.radians(lon)
    scale = x / lon
    y = (
        180.0
        / np.pi
        * np.log(np.tan(np.pi / 4.0 + lat * (np.pi / 180.0) / 2.0))
        * scale
    )
    return x, y


df = pd.read_csv("District_Locations.csv")
df = df.rename(columns={"Long": "lat", "Lat": "lon"})

pop_2024_data = {
    "Colombo": 2375415, "Gampaha": 2436142, "Kalutara": 1305784, "Kandy": 1461895,
    "Matale": 526870, "Nuwaraeliya": 725280, "Galle": 1097372, "Matara": 837889,
    "Hambantota": 671418, "Jaffna": 594751, "Mannar": 123756, "Vavunia": 172312,
    "Mullativu": 122619, "Kilinochchi": 136710, "Batticoloa": 595918, "Ampara": 744551,
    "Trincomalee": 442745, "Kurunegala": 1768156, "Puttalam": 818816, "Anuradhapura": 960080,
    "Polonnaruwa": 447530, "Badulla": 872307, "Moneragala": 527585, "Ratnapura": 1145423,
    "Kegalle": 870476
}

area_data = {
    "Colombo": 699, "Gampaha": 1387, "Kalutara": 1598, "Kandy": 1940, "Matale": 1993,
    "Nuwaraeliya": 1741, "Galle": 1652, "Matara": 1282, "Hambantota": 2609, "Jaffna": 1025,
    "Mannar": 1996, "Vavunia": 1967, "Mullativu": 2617, "Kilinochchi": 1279, "Batticoloa": 2854,
    "Ampara": 4415, "Trincomalee": 2727, "Kurunegala": 4816, "Puttalam": 3072,
    "Anuradhapura": 7179, "Polonnaruwa": 3293, "Badulla": 2861, "Moneragala": 5639,
    "Ratnapura": 3275, "Kegalle": 1693
}

province_data = {
    "Colombo": "Western", "Gampaha": "Western", "Kalutara": "Western",
    "Kandy": "Central", "Matale": "Central", "Nuwaraeliya": "Central",
    "Galle": "Southern", "Matara": "Southern", "Hambantota": "Southern",
    "Jaffna": "Northern", "Kilinochchi": "Northern", "Mannar": "Northern",
    "Vavunia": "Northern", "Mullativu": "Northern", "Batticoloa": "Eastern",
    "Ampara": "Eastern", "Trincomalee": "Eastern", "Kurunegala": "North Western",
    "Puttalam": "North Western", "Anuradhapura": "North Central",
    "Polonnaruwa": "North Central", "Badulla": "Uva", "Moneragala": "Uva",
    "Ratnapura": "Sabaragamuwa", "Kegalle": "Sabaragamuwa"
}

df["population_2024"] = df["District"].map(pop_2024_data)
df["area_sqkm"] = df["District"].map(area_data)
df["province"] = df["District"].map(province_data)
df["density"] = (df["population_2024"] / df["area_sqkm"]).round(2)

x_coords, y_coords = [], []
for lat, lon in zip(df["lat"], df["lon"]):
    x, y = lat_lon_to_mercator(lat, lon)
    x_coords.append(x)
    y_coords.append(y)

df["x"] = x_coords
df["y"] = y_coords
df["size"] = np.sqrt(df["density"]) * 0.9 + 14



sl_min_x, sl_min_y = lat_lon_to_mercator(5.8, 79.2)
sl_max_x, sl_max_y = lat_lon_to_mercator(9.9, 82.2)


source = ColumnDataSource(df)
original_source = ColumnDataSource(df)


p = figure(
    title="SRI LANKA POPULATION DENSITY ANALYTICS (2024)",
    x_range=(sl_min_x, sl_max_x),
    y_range=(sl_min_y, sl_max_y),
    x_axis_type="mercator",
    y_axis_type="mercator",
    height=580,
    width=650,
    background_fill_color="#1a1c23",
    border_fill_color="#1a1c23",
    outline_line_color=None,
    tools="pan,wheel_zoom,box_zoom,reset",
)

p.add_tile("OSM")

color_mapper = LinearColorMapper(
    palette=Plasma256, low=df["density"].min(), high=df["density"].max()
)

p.scatter(
    x="x",
    y="y",
    size="size",
    source=source,
    fill_color={"field": "density", "transform": color_mapper},
    fill_alpha=0.85,
    line_color="#ffffff",
    line_width=1.8,
    hover_fill_color="#00ffff",
    hover_line_color="#ffffff",
)

hover_html = """
    <div style="background-color: #262932; padding: 10px; border-radius: 8px; color: #ffffff; font-family: sans-serif;">
        <h3 style="margin: 0 0 5px 0; color: #00e676; border-bottom: 1px solid #444; padding-bottom: 3px;">@District District</h3>
        <p style="margin: 3px 0; font-size: 13px;"><b>Province:</b> <span style="color: #ffb74d;">@province</span></p>
        <p style="margin: 3px 0; font-size: 13px;"><b>Population (2024):</b> @population_2024{0,0}</p>
        <p style="margin: 3px 0; font-size: 13px;"><b>Density:</b> <span style="color: #00e5ff; font-weight: bold;">@density / sq km</span></p>
        <p style="margin: 3px 0; font-size: 12px; color: #aaa;"><b>Area:</b> @area_sqkm sq km</p>
    </div>
"""
p.add_tools(HoverTool(tooltips=hover_html))

color_bar = ColorBar(
    color_mapper=color_mapper,
    label_standoff=12,
    width=14,
    location=(0, 0),
    background_fill_color="#1a1c23",
    major_label_text_color="#ffffff",
    title="Density",
    title_text_color="#ffffff",
)
p.add_layout(color_bar, "right")

p.xaxis.visible = False
p.yaxis.visible = False
p.title.text_color = "#ffffff"
p.title.text_font_size = "14pt"
p.title.text_font_style = "bold"

# Widgets
district_list = ["All"] + sorted(list(df["District"].unique()))
province_list = ["All"] + sorted(list(df["province"].unique()))

select_district = Select(title="Zoom District:", value="All", options=district_list, width=220)
select_province = Select(title="Filter Province:", value="All", options=province_list, width=220)
slider_density = Slider(start=0, end=3500, value=0, step=100, title="Min Density:", width=220)


callback = CustomJS(
    args=dict(
        source=source,
        original_source=original_source,
        dist_select=select_district,
        prov_select=select_province,
        dens_slider=slider_density,
        x_range=p.x_range,
        y_range=p.y_range,
        sl_min_x=sl_min_x, sl_max_x=sl_max_x,
        sl_min_y=sl_min_y, sl_max_y=sl_max_y
    ),
    code="""
        const data = source.data;
        const orig_data = original_source.data;
        
        const selected_dist = dist_select.value;
        const selected_prov = prov_select.value;
        const min_dens = dens_slider.value;
        
        // Reset Arrays
        for (let key in data) {
            data[key] = [];
        }
        
        // Filtering Data
        for (let i = 0; i < orig_data['District'].length; i++) {
            const matches_prov = (selected_prov === 'All') || (orig_data['province'][i] === selected_prov);
            const matches_dens = orig_data['density'][i] >= min_dens;
            
            if (matches_prov && matches_dens) {
                for (let key in data) {
                    data[key].push(orig_data[key][i]);
                }
            }
        }
        source.change.emit();
        
        // Zoom Logic
        if (selected_dist === 'All') {
            x_range.start = sl_min_x;
            x_range.end = sl_max_x;
            y_range.start = sl_min_y;
            y_range.end = sl_max_y;
        } else {
            for (let i = 0; i < orig_data['District'].length; i++) {
                if (orig_data['District'][i] === selected_dist) {
                    const cx = orig_data['x'][i];
                    const cy = orig_data['y'][i];
                    const offset = 35000;
                    
                    x_range.start = cx - offset;
                    x_range.end = cx + offset;
                    y_range.start = cy - offset;
                    y_range.end = cy + offset;
                    break;
                }
            }
        }
    """
)

select_district.js_on_change('value', callback)
select_province.js_on_change('value', callback)
slider_density.js_on_change('value', callback)


controls_panel = column(select_district, select_province, slider_density, margin=(10, 20, 10, 0))
layout = row(controls_panel, p)

show(layout)

Loading BokehJS ...